# 01 - Dataset Preprocessing

Download Lichess standard rated games across multiple time periods (pre-AI and post-AI),
sample 200k games with fair ELO and period distribution, then extract:

1. **games.csv** - ELO, opening, winner, first moves, castling, game length, sacrifice count, eval volatility
2. **moves.csv** - eval, clock, piece, square, material, captures, checks, sacrifices (eval moves only)
3. **blunders.csv** - moves causing eval drops >= 200cp
4. **piece_squares.csv** - aggregated piece-square counts by period (for heatmaps)
5. **material_curve.csv** - average material at each ply by period

Designed to tell the story of how AI (AlphaZero 2017, Stockfish NNUE 2020) changed human chess.

## Configuration
Set the month to download and processing parameters below.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Time periods anchored around two major AI breakthroughs:
#   AlphaZero (Dec 2017) and Stockfish NNUE (Aug 2020)
PERIODS = [
    {"years": [2014, 2015, 2016], "label": "pre-ai"},
    {"years": [2018, 2019],       "label": "early-post-ai"},
    {"years": [2021, 2022],       "label": "nnue-era"},
    {"years": [2024, 2025],       "label": "modern"},
]

# Months per year to scan (None = all 12)
MONTH = None  # e.g. [1, 6, 12] for specific months

# Total games to collect (evenly split across period x ELO cells)
TOTAL_GAMES = 3_000_000

# ELO brackets for fair distribution across skill levels
ELO_BRACKETS = [
    (0, 1000),
    (1000, 1400),
    (1400, 1800),
    (1800, 2200),
    (2200, 2600),
    (2600, 9999),
]

# Blunder threshold in centipawns (200 = same as Lichess)
BLUNDER_CUTOFF_CP = 200

# Sync parsed results to Drive every N games
SYNC_EVERY = 25_000

# Output file prefix
OUTPUT_PREFIX = "lichess_sampled"

## Setup
Install dependencies and mount Google Drive.

In [ ]:
import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# python-chess: PGN parsing + board state; zstandard: streaming .zst decompression
install("python-chess")
install("zstandard")
install("pandas")
install("tqdm")

print("Dependencies installed.")

In [ ]:
from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

# VM_DATA_DIR is ephemeral (lost on disconnect); Drive is the persistent store
DRIVE_ROOT = "/content/drive/MyDrive/Learning-document/Data Visualization/Project 2"
DRIVE_DATA_DIR = os.path.join(DRIVE_ROOT, "data")
VM_DATA_DIR = "/content/data"

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(VM_DATA_DIR, exist_ok=True)

print(f"Drive root:    {DRIVE_ROOT}")
print(f"Drive data:    {DRIVE_DATA_DIR}")
print(f"VM data dir:   {VM_DATA_DIR}")

## Download
Download the zstd-compressed PGN file from Lichess.

In [ ]:
def download_month(year, month):
    """Download zstd-compressed PGN file for a given month. Returns local path."""
    month_str = f"{year}-{month:02d}"
    filename = f"lichess_db_standard_rated_{month_str}.pgn.zst"
    url = f"https://database.lichess.org/standard/{filename}"
    local_path = f"/content/{filename}"

    # Skip re-download if the file survived from a previous cell run
    if os.path.exists(local_path):
        print(f"File already exists: {local_path}")
        return local_path

    print(f"Downloading {url}...")
    print("This may take 10-30 minutes depending on file size.")
    urllib.request.urlretrieve(url, local_path)
    size_gb = os.path.getsize(local_path) / (1024**3)
    print(f"Downloaded: {size_gb:.1f} GB")
    return local_path

print("download_month() ready.")

## Streaming PGN Parser
Read the compressed PGN file line-by-line (no need to decompress to disk).
Extract both game metadata and move-level eval/clock data.

In [ ]:
import zstandard as zstd
import io
import chess
import chess.pgn
import re
from collections import defaultdict
from tqdm.notebook import tqdm
import pandas as pd
import time
import json
import urllib.request
import statistics

# King is 0 because it's never captured; only used for material imbalance logic
PIECE_VALUES = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
    chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0,
}

def get_material(board):
    """Get (white_material, black_material) from board state."""
    wm = sum(PIECE_VALUES[p.piece_type] for p in board.piece_map().values() if p.color == chess.WHITE)
    bm = sum(PIECE_VALUES[p.piece_type] for p in board.piece_map().values() if p.color == chess.BLACK)
    return wm, bm

print("Libraries loaded.")

In [ ]:
def open_zst_stream(filepath):
    """Open a .zst file for streaming line-by-line reading.
    Uses a generator so only a small buffer is in memory at a time — avoids
    decompressing the full multi-GB file to disk.
    """
    dctx = zstd.ZstdDecompressor()
    with open(filepath, 'rb') as fh:
        stream = dctx.stream_reader(fh)
        text_stream = io.TextIOWrapper(stream, encoding='utf-8')
        yield from text_stream


def read_games_from_stream(lines):
    """Yield individual game strings from a stream of PGN lines.
    A PGN game ends at a blank line that follows movetext (not just headers),
    so we check for at least one non-bracket line before emitting.
    """
    current_game = []
    for line in lines:
        line = line.rstrip('\n')
        if line == '' and current_game:
            has_movetext = any(not l.startswith('[') for l in current_game)
            if has_movetext:
                yield '\n'.join(current_game)
                current_game = []
            else:
                current_game.append(line)
        else:
            if line:
                current_game.append(line)
    if current_game:
        yield '\n'.join(current_game)


def parse_header(headers, key):
    """Safely extract a PGN header value."""
    try:
        return headers.get(key, None)
    except:
        return None


def parse_time_control(tc_str):
    """Parse time control like '300+0' or '600+5' into (base_seconds, increment)."""
    if not tc_str or '+' not in tc_str:
        return None, None
    try:
        parts = tc_str.split('+')
        return int(parts[0]), int(parts[1])
    except:
        return None, None


def extract_elos_from_text(game_text):
    """Quickly extract WhiteElo and BlackElo from raw PGN text.
    Avoids a full chess.pgn parse for the Phase 1 fast-scan where only
    ELO is needed for bucketing.
    """
    white_elo = None
    black_elo = None
    for line in game_text.split('\n'):
        if line.startswith('[WhiteElo "'):
            try: white_elo = int(line.split('"')[1])
            except: pass
        elif line.startswith('[BlackElo "'):
            try: black_elo = int(line.split('"')[1])
            except: pass
        if white_elo is not None and black_elo is not None:
            break
    return white_elo, black_elo


print("Parser functions defined.")